# RCM check 3 - At which nesting step do the RCM differences appear? EC-Earth3, RegCM5 0.75 deg and 0.25 deg

**Ad hoc test, outside the chain**: it reads the cache of notebook 1 (EC-Earth3 and RegCM5 0.25 deg monthly means of
the 2050 pair), the monthly files of the RegCM5 0.75 deg runs prepared by ECOADAPT
(`ECOADAPT/AMOC/outputs/preproc/075/<run>/mon`, read only) and E-OBS regridded by ECOADAPT onto the 0.75 deg grid
(`obs/regridded/075`, 2000-2024), and writes only figures / tables in `outputs/` next to this notebook.

**Question.** The RCM is nested twice: EC-Earth3 -> RegCM5 0.75 deg -> RegCM5 0.25 deg. Notebooks 1 and 2 found, at
0.25 deg: (1) a winter precipitation about two thirds lower than that of EC-Earth3 over the British Isles, France and
Iberia, and higher in the east; (2) a stronger autumn cooling of the east and the north under hosing; (3) a drying
under hosing about 1.8 times stronger. Does each difference exist already in the first nest (0.75 deg, driven
directly by the GCM: RegCM physics or boundary conditions), or does it appear in the second (0.75 -> 0.25 deg)?
ECOADAPT also found that the two nests disagree on the effect of the AMOC on summer exposure (+29.5 against +7.6
points).

**Method.** As notebooks 1 and 2: 2050 pair, years `YEARS` (2130-2149), both RCMs aggregated onto the EC-Earth3 grid
(mean of the RCM cells whose centre falls in a GCM cell, relaxation zones left out), the cells covered by both RCMs in
the European LISFLOOD domain, the 8 PRUDENCE regions. For each variable and season: (a) the climate of each RCM
against its driver (no-hosing run); (b) the hosing change of each model with 90 % bootstrap intervals, and the
agreement across regions of each pair of models. (c) E-OBS 2000-2024 per region as an observed reference: it is the
present climate, while the runs are the stabilised 2050 climate, so the temperatures of the runs should be a few K
above it; for precipitation a difference of tens of percent cannot come from the climate change alone.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, glob, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from scipy import sparse
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
ECO = Path(os.environ.get("ECOADAPT_ROOT", Path(cfg["data_root"]).parent / "ECOADAPT" / "AMOC"))   # read only
R075_RUNS = {"nohos": "stabilization-ssp585-2050", "hos": "stabilization-ssp585-2050-hosing"}
RCM_TAGS = {"nohos": "2050_Base", "hos": "2050_Hose"}                     # cache of notebook 1 (RegCM5 0.25)
GCM_RUNS = {"nohos": "gcm_2050_nohosing", "hos": "gcm_2050_hosing"}
EOBS = {"tasmax": "tx", "tasmin": "tn", "pr": "rr"}
VARS = ["tasmax", "tasmin", "pr"]
DATA_YEARS = (2125, 2149)
YEARS = tuple(int(y) for y in os.environ.get("SIGNAL_YEARS", "2130,2149").split(","))
SPONGE = {"r025": 11 + 2, "r075": 11 + 2}     # cells left out along each RCM edge (relaxation zone + 2)
MIN_COVER = 0.5
SEASONS = {"DJF": (12, 1, 2), "MAM": (3, 4, 5), "JJA": (6, 7, 8), "SON": (9, 10, 11), "year": tuple(range(1, 13))}
PRUDENCE = {"BI": ("British Isles", (50, 59), (-10, 2)), "IP": ("Iberian Peninsula", (36, 44), (-10, 3)),
            "FR": ("France", (44, 50), (-5, 5)), "ME": ("Mid-Europe", (48, 55), (2, 16)),
            "SC": ("Scandinavia", (55, 70), (5, 30)), "AL": ("Alps", (44, 48), (5, 15)),
            "MD": ("Mediterranean", (36, 44), (3, 25)), "EA": ("Eastern Europe", (44, 55), (16, 30))}
N_BOOT = 1000
SEED = 20261120
MODELS = {"gcm": "EC-Earth3", "r075": "RegCM5 0.75", "r025": "RegCM5 0.25"}

CACHE = bu.out_dir(cfg, "rcm_check", "cache")
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
TAG = f"{YEARS[0]}-{YEARS[1]}"
print(f"years {TAG} | cache of notebook 1: {CACHE} | ECOADAPT: {ECO}")

## 1. Monthly means: EC-Earth3 and RegCM5 0.25 deg from the cache of notebook 1, RegCM5 0.75 deg and E-OBS from ECOADAPT

In [ ]:
DAYS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], float)


def cached(path):
    assert path.exists(), f"missing {path.name}: run 1_hosing_signal_rcm_vs_gcm.ipynb first"
    return xr.open_dataarray(path).load()


def latlon2d(ds):
    for la, lo in (("lat", "lon"), ("xlat", "xlon"), ("latitude", "longitude")):
        if la in ds.variables and lo in ds.variables:
            a, o = ds[la].values.astype(float), ds[lo].values.astype(float)
            if a.ndim == 1:
                o, a = np.meshgrid(o, a)
            return a, ((o + 180.0) % 360.0) - 180.0
    raise KeyError(f"no lat / lon (variables: {list(ds.variables)})")


def months_of(ds):
    # (year, month) of each step (files opened with use_cftime): middle of the time bounds if present, robust to
    # stamps at the start or the end of the month
    bn = [b for b in (ds["time"].attrs.get("bounds"), "time_bnds", "time_bounds") if b and b in ds]
    mid = [b0 + (b1 - b0) / 2 for b0, b1 in ds[bn[0]].values] if bn else list(ds["time"].values)
    return np.array([m.year for m in mid]), np.array([m.month for m in mid])


def to_units(a, units, var, monthly_total_check=False, months=None):
    u = str(units).lower().replace(" ", "")
    if var == "pr":
        if u in ("kgm-2s-1", "kgm**-2s**-1", "kg/m2/s", "mm/s", "kgm^-2s^-1"):
            return a * 86400.0
        if monthly_total_check and np.nanmean(a) > 15:            # monthly totals (mm/month) -> mm/day
            return a / DAYS[months - 1][:, None, None]
        return a
    return a - 273.15 if u in ("k", "kelvin") else a


def r075_monthly(run, var):
    fs = sorted(glob.glob(str(ECO / "outputs" / "preproc" / "075" / R075_RUNS[run] / "mon" / f"{var}_EUR-075_*_{R075_RUNS[run]}_*_mon_*.nc")))
    assert len(fs) >= 1, f"no 0.75 deg monthly file for {run} {var}"
    ds = xr.open_mfdataset(fs, combine="by_coords", use_cftime=True) if len(fs) > 1 else xr.open_dataset(fs[0], use_cftime=True)
    yr, mo = months_of(ds)
    keep = (yr >= DATA_YEARS[0]) & (yr <= DATA_YEARS[1])
    a = ds[var].squeeze(drop=True).values[keep].astype(float)
    a = to_units(a, ds[var].attrs.get("units", ""), var)
    lat, lon = latlon2d(ds)
    t = pd.DatetimeIndex([pd.Timestamp(int(y), int(m), 1) for y, m in zip(yr[keep], mo[keep])])
    assert t.size == 12 * (DATA_YEARS[1] - DATA_YEARS[0] + 1) and t.is_monotonic_increasing, f"{run} {var}: {t.size} months"
    return xr.DataArray(a, dims=("time", "y", "x"), coords={"time": t, "lat": (("y", "x"), lat), "lon": (("y", "x"), lon)}, name=var), \
           ds[var].attrs.get("cell_methods", "")


def eobs_monthly(var):
    f = ECO / "obs" / "regridded" / "075" / f"eobs_{EOBS[var]}_2000-2024_EUR-075_mon.nc"
    if not f.exists():
        return None
    ds = xr.open_dataset(f, use_cftime=True)
    v = [k for k in ds.data_vars if ds[k].ndim >= 3][0]
    yr, mo = months_of(ds)
    a = to_units(ds[v].squeeze(drop=True).values.astype(float), ds[v].attrs.get("units", ""), var, monthly_total_check=True, months=mo)
    lat, lon = latlon2d(ds)
    t = pd.DatetimeIndex([pd.Timestamp(int(y), int(m), 1) for y, m in zip(yr, mo)])
    return xr.DataArray(a, dims=("time", "y", "x"), coords={"time": t, "lat": (("y", "x"), lat), "lon": (("y", "x"), lon)}, name=var)


GCM = {(run, v): cached(CACHE / f"ecearth3_{GCM_RUNS[run]}_{v}_monthly_{DATA_YEARS[0]}-{DATA_YEARS[1]}.nc") for run in GCM_RUNS for v in VARS}
NAT = {("r025", run, v): cached(CACHE / f"rcm025_{RCM_TAGS[run]}_{v}_monthly_native_{DATA_YEARS[0]}-{DATA_YEARS[1]}.nc")
       for run in RCM_TAGS for v in VARS}
CM = {}
for run in R075_RUNS:
    for v in VARS:
        NAT["r075", run, v], CM[run, v] = r075_monthly(run, v)
OBS = {v: eobs_monthly(v) for v in VARS}
print("0.75 deg cell_methods:", {k: v for k, v in CM.items() if k[0] == "nohos"})
print("E-OBS:", {v: (None if o is None else f"{o.sizes['time']} months, grid {o.sizes['y']} x {o.sizes['x']}") for v, o in OBS.items()})

# check that the 0.75 deg tasmax / tasmin are means of daily values (not monthly extremes): daily range against 0.25 deg
G0 = GCM["nohos", "tasmax"]
GLAT, GLON = G0.lat.values, G0.lon.values
dtr = {m: float(np.nanmean(NAT[m, "nohos", "tasmax"].values - NAT[m, "nohos", "tasmin"].values)) for m in ("r025", "r075")}
print(f"mean tasmax - tasmin over the whole RCM grid: 0.25 deg {dtr['r025']:.1f} K, 0.75 deg {dtr['r075']:.1f} K")
assert dtr["r075"] < 1.6 * dtr["r025"], "0.75 deg tasmax / tasmin look like monthly extremes, not means of daily values: use the day/ files"

## 2. Aggregation onto the EC-Earth3 grid, cells covered by both RCMs, PRUDENCE regions

In [ ]:
def cell_area_km2(lat, lon):
    R = 6371.0
    la, lo = np.deg2rad(lat), np.deg2rad(lon)
    d = lambda a1, o1, a2, o2: 2 * R * np.arcsin(np.sqrt(np.sin((a2 - a1) / 2) ** 2
                                                       + np.cos(a1) * np.cos(a2) * np.sin((o2 - o1) / 2) ** 2))
    dx = d(la[:, :-1], lo[:, :-1], la[:, 1:], lo[:, 1:]); dx = np.c_[dx, dx[:, -1:]]
    dy = d(la[:-1], lo[:-1], la[1:], lo[1:]); dy = np.r_[dy, dy[-1:]]
    return dx * dy


ye, xe = bu._edges(GLAT), bu._edges(GLON)
NG = GLAT.size * GLON.size
g_area = (np.diff(np.sin(np.deg2rad(np.clip(ye, -90, 90))))[:, None] * np.deg2rad(np.diff(xe))[None, :]) * 6371.0 ** 2


def aggregator(da, sponge):
    # RCM (or E-OBS) grid -> EC-Earth3 grid: mean of the cells whose centre falls in a GCM cell; share of each GCM cell covered
    lat, lon = da["lat"].values.astype(float), ((da["lon"].values.astype(float) + 180.0) % 360.0) - 180.0
    ny, nx = lat.shape
    inner = np.zeros((ny, nx), bool); inner[sponge:ny - sponge, sponge:nx - sponge] = True
    iy, ix = np.searchsorted(ye, lat) - 1, np.searchsorted(xe, lon) - 1
    ok = inner & (iy >= 0) & (iy < GLAT.size) & (ix >= 0) & (ix < GLON.size)
    col = np.flatnonzero(ok); row = (iy * GLON.size + ix).ravel()[col]
    ones = sparse.csr_matrix((np.ones(col.size), (row, col)), shape=(NG, ny * nx))
    cover = (np.bincount(row, weights=cell_area_km2(lat, lon).ravel()[col], minlength=NG) / g_area.ravel()).reshape(GLAT.size, GLON.size)

    def to_gcm(a):
        flat = np.asarray(a, float).reshape(-1, ny * nx)
        fin = np.isfinite(flat)
        num = ones @ np.where(fin, flat, 0.0).T
        den = ones @ fin.T.astype(float)
        return np.where(den > 0, num / np.where(den > 0, den, 1), np.nan).T.reshape(flat.shape[0], GLAT.size, GLON.size)
    return to_gcm, cover


def nearest(da, sponge, valid=None, max_deg=0.6):
    # grids about as coarse as EC-Earth3 (0.75 deg RCM, E-OBS on its grid): value of the nearest cell (outside the
    # relaxation zone, and valid) within max_deg of the GCM cell centre; the cell-centre mean would leave one GCM cell
    # in two empty (checkerboard). "cover" = 1 where a cell was found.
    from scipy.spatial import cKDTree
    lat, lon = da["lat"].values.astype(float), ((da["lon"].values.astype(float) + 180.0) % 360.0) - 180.0
    ny, nx = lat.shape
    inner = np.zeros((ny, nx), bool); inner[sponge:ny - sponge, sponge:nx - sponge] = True
    if valid is not None:
        inner &= valid
    src = np.flatnonzero(inner.ravel())
    xyz = lambda la, lo: np.c_[np.cos(np.deg2rad(la)) * np.cos(np.deg2rad(lo)), np.cos(np.deg2rad(la)) * np.sin(np.deg2rad(lo)), np.sin(np.deg2rad(la))]
    gla, glo = np.meshgrid(GLAT, GLON, indexing="ij")
    dist, k = cKDTree(xyz(lat.ravel()[src], lon.ravel()[src])).query(xyz(gla.ravel(), glo.ravel()))
    ok = np.rad2deg(2 * np.arcsin(np.minimum(dist / 2, 1))) <= max_deg
    take = src[k]

    def to_gcm(a):
        flat = np.asarray(a, float).reshape(-1, ny * nx)[:, take]
        flat[:, ~ok] = np.nan
        return flat.reshape(-1, GLAT.size, GLON.size)
    return to_gcm, ok.reshape(GLAT.size, GLON.size).astype(float)


AGG = {"r025": aggregator(NAT["r025", "nohos", "tasmax"], SPONGE["r025"]), "r075": nearest(NAT["r075", "nohos", "tasmax"], SPONGE["r075"])}
DATA = {("gcm", run, v): GCM[run, v].assign_coords(lat=GLAT, lon=GLON) for run in GCM_RUNS for v in VARS}
for (m, run, v), da in NAT.items():
    DATA[m, run, v] = xr.DataArray(AGG[m][0](da.values), dims=("time", "lat", "lon"), coords={"time": da.time, "lat": GLAT, "lon": GLON})
OBS_G = {}
for v, o in OBS.items():
    if o is not None:
        # E-OBS regridded by ECOADAPT: the sea is filled with a constant (0) instead of missing values; cells whose value
        # never changes are left out
        valid = np.isfinite(o.values).all(0) & (np.nanstd(o.values, axis=0) > 0)
        f, cov = nearest(o, 0, valid=valid)
        OBS_G[v] = xr.DataArray(f(o.values), dims=("time", "lat", "lon"), coords={"time": o.time, "lat": GLAT, "lon": GLON})
        print(f"E-OBS {v}: {valid.sum()} valid cells of {valid.size}, {int(cov.sum())} EC-Earth3 cells with an E-OBS value")

DOM = (bu.load_domain_mask(cfg, "ecearth3", like=G0.isel(time=0)).values
       & (AGG["r025"][1] >= MIN_COVER) & (AGG["r075"][1] >= MIN_COVER))
LAT2 = np.broadcast_to(GLAT[:, None], DOM.shape); LON2 = np.broadcast_to(GLON[None, :], DOM.shape)
WREG = {}
for r, (name, la, lo) in PRUDENCE.items():
    msk = DOM & (LAT2 >= la[0]) & (LAT2 <= la[1]) & (LON2 >= lo[0]) & (LON2 <= lo[1])
    w = np.where(msk, np.cos(np.deg2rad(LAT2)), 0.0).ravel()
    WREG[r] = w / w.sum()
    print(f"{r} {name:18s}: {msk.sum():3d} EC-Earth3 cells")
w = np.where(DOM, np.cos(np.deg2rad(LAT2)), 0.0).ravel(); WREG["DOMAIN"] = w / w.sum()
REGIONS = list(PRUDENCE) + ["DOMAIN"]
print(f"GCM cells covered by both RCMs in the domain: {DOM.sum()}")

## 3. Climate of each model per region (no-hosing run, `YEARS`), and E-OBS 2000-2024

In [ ]:
def season_means(da, months, years):
    t = pd.DatetimeIndex(da["time"].values)
    djf = len(months) == 3 and 12 in months
    sy = t.year + (djf & (t.month == 12)).astype(int)
    v, out = da.values, []
    for y in range(years[0], years[1] + 1):
        k = np.flatnonzero(np.isin(t.month, months) & (sy == y))
        if len(k) == len(months):
            wt = DAYS[t.month[k] - 1]
            out.append(np.tensordot(wt / wt.sum(), v[k], axes=1))
    return np.stack(out)


def regional(da, months, years=YEARS):
    # weighted mean over the cells of each region that have a value (E-OBS has none over the sea)
    x = season_means(da, months, years).reshape(-1, DOM.size)
    fin = np.isfinite(x)
    x0 = np.where(fin, x, 0.0)
    return np.stack([(x0 @ WREG[r]) / (fin @ WREG[r]) for r in REGIONS], axis=1)


def diff(a, b, var):
    return 100.0 * (a / b - 1.0) if var == "pr" else a - b


rows = []
for var in VARS:
    for s, months in SEASONS.items():
        val = {m: regional(DATA[m, "nohos", var], months).mean(0) for m in MODELS}
        obs = regional(OBS_G[var], months, (2001, 2024)).mean(0) if var in OBS_G else np.full(len(REGIONS), np.nan)
        for i, r in enumerate(REGIONS):
            rows.append(dict(var=var, season=s, region=r, unit="%" if var == "pr" else "K", gcm=val["gcm"][i],
                             r075_minus_gcm=diff(val["r075"][i], val["gcm"][i], var),
                             r025_minus_gcm=diff(val["r025"][i], val["gcm"][i], var),
                             r025_minus_r075=diff(val["r025"][i], val["r075"][i], var), eobs=obs[i],
                             gcm_minus_eobs=diff(val["gcm"][i], obs[i], var), r075_minus_eobs=diff(val["r075"][i], obs[i], var),
                             r025_minus_eobs=diff(val["r025"][i], obs[i], var)))
CLIM = pd.DataFrame(rows)
CLIM.to_csv(OUT / f"nest_climate_{TAG}.csv", index=False)
for var in VARS:
    u = "%" if var == "pr" else "K"
    t = CLIM[CLIM["var"] == var]
    print(f"\n{var}: RCM 0.75 - GCM / RCM 0.25 - GCM ({u}), no hosing {TAG}")
    show = t.assign(v=[f"{a:+.1f} / {b:+.1f}" for a, b in zip(t.r075_minus_gcm, t.r025_minus_gcm)])
    display(show.pivot(index="region", columns="season", values="v").loc[REGIONS, list(SEASONS)])
    if var in OBS_G:
        print(f"{var}: model - E-OBS 2000-2024 ({u}): GCM / RCM 0.75 / RCM 0.25 (the runs are the stabilised 2050 climate)")
        show = t.assign(v=[f"{a:+.1f} / {b:+.1f} / {c:+.1f}" for a, b, c in zip(t.gcm_minus_eobs, t.r075_minus_eobs, t.r025_minus_eobs)])
        display(show.pivot(index="region", columns="season", values="v").loc[REGIONS, list(SEASONS)])

## 4. Hosing response per region: the three models, 90 % bootstrap intervals, agreement across regions

In [ ]:
def change(hos, nohos, var):
    return 100.0 * (hos / nohos - 1.0) if var == "pr" else hos - nohos


rng = np.random.default_rng(SEED)
rows, CH = [], {}
for var in VARS:
    for s, months in SEASONS.items():
        Y = {(m, run): regional(DATA[m, run, var], months) for m in MODELS for run in ("nohos", "hos")}
        n = Y["gcm", "hos"].shape[0]
        d = {m: change(Y[m, "hos"].mean(0), Y[m, "nohos"].mean(0), var) for m in MODELS}
        boot = {m: [] for m in MODELS}
        for _ in range(N_BOOT):
            pick = {k: Y[k][rng.integers(0, n, n)].mean(0) for k in Y}
            for m in MODELS:
                boot[m].append(change(pick[m, "hos"], pick[m, "nohos"], var))
        boot = {m: np.array(b) for m, b in boot.items()}
        CH[var, s] = d
        q = lambda a: np.quantile(a, [0.05, 0.95], axis=0)
        for i, r in enumerate(REGIONS):
            row = dict(var=var, season=s, region=r, unit="%" if var == "pr" else "K")
            for m in MODELS:
                lo, hi = q(boot[m])
                row.update({m: d[m][i], f"{m}_lo": lo[i], f"{m}_hi": hi[i]})
            for a_, b_ in (("r075", "gcm"), ("r025", "gcm"), ("r025", "r075")):
                lo, hi = q(boot[a_] - boot[b_])
                row[f"{a_}_vs_{b_}_differs"] = bool(lo[i] > 0 or hi[i] < 0)
            rows.append(row)
    print(f"{var}: done")
REG = pd.DataFrame(rows)
REG.to_csv(OUT / f"nest_changes_{TAG}.csv", index=False)

rows = []
reg8 = [i for i, r in enumerate(REGIONS) if r != "DOMAIN"]
for var in VARS:
    for s in SEASONS:
        d = CH[var, s]
        row = dict(var=var, season=s, **{f"mean_{m}": d[m][reg8].mean() for m in MODELS})
        for a_, b_ in (("r075", "gcm"), ("r025", "gcm"), ("r025", "r075")):
            row[f"r_{a_}_{b_}"] = np.corrcoef(d[b_][reg8], d[a_][reg8])[0, 1]
            row[f"n_{a_}_{b_}_differ"] = int(REG[(REG["var"] == var) & (REG["season"] == s) & (REG["region"] != "DOMAIN")][f"{a_}_vs_{b_}_differs"].sum())
        rows.append(row)
AGR = pd.DataFrame(rows)
AGR.to_csv(OUT / f"nest_agreement_{TAG}.csv", index=False)
with pd.option_context("display.float_format", "{:.2f}".format, "display.width", 250, "display.max_columns", None):
    display(AGR)
for var in VARS:
    t = REG[REG["var"] == var]
    print(f"\n{var}: hosing change GCM / RCM 0.75 / RCM 0.25 ({'%' if var == 'pr' else 'K'})")
    show = t.assign(v=[f"{a:+.2f} / {b:+.2f} / {c:+.2f}" for a, b, c in zip(t.gcm, t.r075, t.r025)])
    display(show.pivot(index="region", columns="season", values="v").loc[REGIONS, list(SEASONS)])

## 5. Figures

In [ ]:
BORDERS = bu.borders(cfg, "10M")
COL = {"gcm": "k", "r075": "C1", "r025": "C3"}

# regional changes of the two RCMs against EC-Earth3
fig, axes = plt.subplots(2, len(VARS), figsize=(5 * len(VARS), 9))
for j, var in enumerate(VARS):
    for i, m in enumerate(("r075", "r025")):
        ax = axes[i, j]
        for k, s in enumerate(SEASONS):
            ax.scatter(CH[var, s]["gcm"][reg8], CH[var, s][m][reg8], s=22, color=f"C{k}", label=s)
        lim = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lim, lim, "k--", lw=0.7); ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel("EC-Earth3"); ax.set_ylabel(MODELS[m]); ax.set_title(f"{var}: regional hosing change ({'%' if var == 'pr' else 'K'})", fontsize=9)
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / f"nest_scatter_{TAG}.png", dpi=110); plt.show()

# precipitation climate of the two RCMs against EC-Earth3, DJF and JJA
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for i, s in enumerate(("DJF", "JJA")):
    g = season_means(DATA["gcm", "nohos", "pr"], SEASONS[s], YEARS).mean(0)
    for j, m in enumerate(("r075", "r025")):
        r = season_means(DATA[m, "nohos", "pr"], SEASONS[s], YEARS).mean(0)
        ax = axes[i, j]
        pm = ax.pcolormesh(GLON, GLAT, np.where(DOM, 100.0 * (r / g - 1.0), np.nan), shading="nearest", cmap="BrBG", vmin=-80, vmax=80)
        bu.draw_borders(ax, BORDERS, lw=0.3); ax.set_xlim(-12, 45); ax.set_ylim(34, 72)
        ax.set_title(f"pr {s}: {MODELS[m]} / EC-Earth3 - 1 (%), no hosing {TAG}", fontsize=9)
        plt.colorbar(pm, ax=ax, shrink=0.8)
plt.tight_layout(); plt.savefig(OUT / f"nest_pr_climate_{TAG}.png", dpi=110); plt.show()

# SON tasmax hosing change of the three models
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
for ax, m in zip(axes, MODELS):
    c = season_means(DATA[m, "hos", "tasmax"], SEASONS["SON"], YEARS).mean(0) - season_means(DATA[m, "nohos", "tasmax"], SEASONS["SON"], YEARS).mean(0)
    pm = ax.pcolormesh(GLON, GLAT, np.where(DOM, c, np.nan), shading="nearest", cmap="RdBu_r", vmin=-2.5, vmax=2.5)
    bu.draw_borders(ax, BORDERS, lw=0.3); ax.set_xlim(-12, 45); ax.set_ylim(34, 72)
    ax.set_title(f"tasmax SON hosing - no hosing (K): {MODELS[m]}", fontsize=9)
    plt.colorbar(pm, ax=ax, shrink=0.8)
plt.tight_layout(); plt.savefig(OUT / f"nest_tasmax_SON_change_{TAG}.png", dpi=110); plt.show()

## 6. Biases against E-OBS over the same years: EC-Earth3 historical, its change, and an estimate for the RCMs

The comparison of section 3 mixes the bias of each model with the climate change between 2000-2024 (E-OBS) and the
stabilised 2050 climate (the runs). Here: (a) the bias of EC-Earth3 against E-OBS over the same years `HIST_YEARS`
(overlap of E-OBS and of the historical run); (b) the change of EC-Earth3 from `HIST_YEARS` to the no-hosing run
`YEARS`; (c) an estimate of the bias of each RCM, assuming that its change is that of its driver (there is no
historical RCM run): its future climate against E-OBS shifted by the change of EC-Earth3, i.e. RCM - GCM + bias of
the GCM (temperature), (RCM / GCM) x (1 + bias of the GCM) - 1 (pr). Notebook 1 found the same annual hosing response
in both models but different seasonal responses: the estimate is the more uncertain, the more the RCM changes
differently from its driver.

In [ ]:
HIST_YEARS = (2000, 2014)        # E-OBS 2000-2024, EC-Earth3 historical to 2014
REGION = {"lat": (GLAT.min() - 1.0, GLAT.max() + 1.0), "lon": (GLON.min() - 1.0, GLON.max() + 1.0)}


def gcm_hist_monthly(var):
    path = CACHE / f"ecearth3_gcm_historical_{var}_monthly_{HIST_YEARS[0]}-{HIST_YEARS[1]}.nc"
    if path.exists():
        m = xr.open_dataarray(path).load()
    else:
        d = bu.gcm_ba_input(cfg, "gcm_historical", var, (str(HIST_YEARS[0]), str(HIST_YEARS[1])), REGION)
        m = d.resample(time="MS").mean().load()
        m = m.assign_coords(time=pd.DatetimeIndex([pd.Timestamp(t.year, t.month, 1) for t in m.indexes["time"]]))
        m.name = var
        tmp = path.with_suffix(".tmp.nc"); m.to_netcdf(tmp); tmp.replace(path)
    m = m.sel(lat=GLAT, lon=GLON, method="nearest")
    assert np.allclose(m.lat.values, GLAT, atol=0.01) and np.allclose(m.lon.values, GLON, atol=0.01), "historical grid differs"
    return m.assign_coords(lat=GLAT, lon=GLON)


rows = []
for var in [v for v in VARS if v in OBS_G]:
    H = gcm_hist_monthly(var)
    for s, months in SEASONS.items():
        obs = regional(OBS_G[var], months, HIST_YEARS).mean(0)
        gh = regional(H, months, HIST_YEARS).mean(0)
        fut = {m: regional(DATA[m, "nohos", var], months).mean(0) for m in MODELS}
        for i, r in enumerate(REGIONS):
            if var == "pr":
                bias = 100.0 * (gh[i] / obs[i] - 1.0); chg = 100.0 * (fut["gcm"][i] / gh[i] - 1.0)
                est = {m: 100.0 * (fut[m][i] / fut["gcm"][i] * gh[i] / obs[i] - 1.0) for m in ("r075", "r025")}
            else:
                bias = gh[i] - obs[i]; chg = fut["gcm"][i] - gh[i]
                est = {m: fut[m][i] - fut["gcm"][i] + bias for m in ("r075", "r025")}
            rows.append(dict(var=var, season=s, region=r, unit="%" if var == "pr" else "K", eobs=obs[i], gcm_hist=gh[i],
                             gcm_bias=bias, gcm_change=chg, r075_est_bias=est["r075"], r025_est_bias=est["r025"]))
BIAS = pd.DataFrame(rows)
BIAS.to_csv(OUT / f"nest_bias_estimate_{TAG}.csv", index=False)
for var in BIAS["var"].unique():
    t = BIAS[BIAS["var"] == var]
    u = "%" if var == "pr" else "K"
    print(f"\n{var} ({u}): bias of EC-Earth3 against E-OBS {HIST_YEARS[0]}-{HIST_YEARS[1]} / change of EC-Earth3 to {TAG} / "
          f"estimated bias RCM 0.75 / estimated bias RCM 0.25")
    show = t.assign(v=[f"{a:+.1f} / {b:+.1f} / {c:+.1f} / {d:+.1f}" for a, b, c, d in
                       zip(t.gcm_bias, t.gcm_change, t.r075_est_bias, t.r025_est_bias)])
    display(show.pivot(index="region", columns="season", values="v").loc[REGIONS, list(SEASONS)])

## 7. Reading

- **Climate (section 3).** A difference RCM - GCM already present at 0.75 deg comes from RegCM itself (physics, or the
  boundary conditions it receives from EC-Earth3); a difference that appears only at 0.25 deg (`r025_minus_r075`)
  comes from the second nesting. The model - E-OBS columns of section 3 mix the bias and the climate change (present
  observations against the stabilised 2050 climate): use the estimates of section 6, made over the same years.
- **Biases (section 6).** EC-Earth3 against E-OBS over the same years, and the estimated bias of each RCM under the
  assumption that it changes as its driver.
- **Hosing response (section 4).** For each difference found in notebooks 1-2 (autumn cooling of the east and the
  north, stronger drying): is it already in the 0.75 deg run (`r_r075_gcm`, `n_r075_gcm_differ`) or only at 0.25 deg
  (`r_r025_r075`, `n_r025_r075_differ`)?